In [0]:
import os
import mlflow
import mlflow.spark
from mlflow.models import infer_signature
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté

# Lecture du Batch Silver
df = spark.read.table("main.silver.transactions_silver")

# Séparer features et label
feature_cols = [f"V{i}" for i in range(1, 29)] + ["amount"]
label_col = "is_fraud"

# VectorAssembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df).select("features", label_col)

# Séparer jeux pour train / test
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# Choix du modèle Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol=label_col)
model = lr.fit(train_df)

# Obligatoire en Free Edition
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/main/ml/models_volume/tmp"  # noqa: E501
dbutils.fs.mkdirs("/Volumes/main/ml/models_volume/tmp")

mlflow.set_experiment("/Users//phamgerard69@gmail.com/fraud_detection_experiment")  # noqa: E501

# Créer la signature en inférant depuis un échantillon
sample_input = test_df.limit(5).toPandas()
sample_output = model.transform(test_df.limit(5)).select("prediction").toPandas()
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run() as run:

    # Entraînement
    model = lr.fit(train_df)

    # Evaluation
    pred = model.transform(test_df)
    pred.select("prediction", "probability", label_col).show()
    auc_eval = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")  # noqa: E501
    auc = auc_eval.evaluate(pred)

    acc_eval = MulticlassClassificationEvaluator(labelCol=label_col, metricName="accuracy")  # noqa: E501
    accuracy = acc_eval.evaluate(pred)

    f1_eval = MulticlassClassificationEvaluator(labelCol=label_col, metricName="f1")  # noqa: E501
    f1 = f1_eval.evaluate(pred)

    print("AUC:", auc)
    print("Accuracy:", accuracy)
    print("F1-score:", f1)

    # Confusion matrix
    cm = (
        pred.groupBy(label_col, "prediction")
            .count()
            .orderBy(label_col, "prediction")
    )

    # Logging métriques
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1", f1)

    # Logging confusion matrix
    mlflow.log_table(cm.toPandas(), "confusion_matrix.json")

    # Log et enregistrer le modèle dans Unity Catalog
    model_info = mlflow.spark.log_model(
        model,
        artifact_path="fraud_model",
        dfs_tmpdir="/Volumes/main/ml/models_volume/tmp",
        signature=signature,  # Signature requise pour UC
        registered_model_name="main.ml.fraud_detection_model"  # Enregistrement UC
    )

    print("✓ Modèle enregistré: main.ml.fraud_detection_model")
    print(f"  Run ID: {run.info.run_id}")
    print(f"  Model URI: {model_info.model_uri}")